# Theta phase entrainment of hippocampal neurons (DANDI:000059)

Hippocampal neurons do not fire at arbitrary moments during locomotion. Their spikes
are timed relative to the phase of the 5-11 Hz theta oscillation in the local field
potential, a phenomenon usually called theta phase locking or theta entrainment. This
notebook demonstrates that entrainment in real data, quantifies it against an explicit
null model, and checks it against several alternative explanations.

**Dataset.** DANDI:000059, "Cooling of Medial Septum Reveals Theta Phase Lag
Coordination of Hippocampal Cell Assemblies" (Petersen & Buzsáki, *Neuron* 2020).
Rats run a spatial alternation maze while silicon probes record the hippocampus, and
on a subset of trials the medial septum is cooled through an implanted Peltier device.
Cooling slows the theta rhythm without stopping it, which provides a causal
manipulation of the oscillation inside the same recording.

Each session is distributed as two NWB assets: a raw-ephys file that also carries a
1250 Hz LFP series, and a processed file with spike-sorted units, position, running
speed, septal temperature and trial intervals. Both are streamed from S3 with
`remfile` and a local disk cache; nothing is downloaded in full.

**What is measured.** For each unit, the theta phase of the LFP at the time of every
spike during running. The strength of entrainment is the mean resultant length (MRL)
of that circular distribution, and its preferred phase is the circular mean. Phase 0
is the peak of the band-passed LFP on the channel the original authors flagged as the
theta reference; phase 180° is its trough.

**Controls applied.** Chance level for each unit comes from circularly shifting that
unit's own spike train within the running epochs, which preserves its spike count and
inter-spike-interval structure while destroying its alignment to the LFP. Locking is
further checked for split-half stability, for consistency with the spike-triggered LFP
average, and (with a Poisson GLM) for whether theta phase still predicts spiking once
running speed and the unit's own spike history are accounted for.

## Setup

The analysis is split across two helper modules that live next to this notebook:
`theta_lib.py` (streaming, loading, filtering, circular statistics) and `analysis.py`
(per-session phase-locking pipeline). The numbered scripts `01_…` to `06_…` are the
same code organised as a runnable pipeline; this notebook calls into them so that
figures and numbers cannot drift apart.

In [1]:
import os
import pickle
import subprocess

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pynapple as nap

import analysis as an
import theta_lib as tl

pd.set_option("display.width", 140)
print("sessions used:")
for s in tl.SESSIONS:
    print("  ", s)
print(f"\ntheta band: {tl.THETA_BAND} Hz | running threshold: {tl.SPEED_THRESHOLD} cm/s "
      f"| minimum spikes per unit: {tl.MIN_SPIKES}")

sessions used:
   Peter-MS21-180628-155921-concat
   Peter-MS21-180629-110332-concat
   Peter-MS22-180629-110319-concat
   Peter-MS21-180625-153927-concat
   Peter-MS13-171128-113924-concat

theta band: (5.0, 11.0) Hz | running threshold: 5.0 cm/s | minimum spikes per unit: 200


In [2]:
def run_step(script):
    """Run one pipeline script, streaming its output."""
    print(f"--- {script} ---")
    subprocess.run(["python", script], check=True)

## 1. Load one session and validate every data stream

Before any analysis, each stream is inspected: the two NWB assets are checked for a
shared clock (the last spike time must match the LFP duration), the LFP is plotted
next to its band-passed version and its Hilbert phase, spikes are rastered against
it, and running speed and septal temperature are plotted over the session.

In [3]:
run_step("01_load_data.py")

--- 01_load_data.py ---


/Users/bdichter/dev/agent-foundational-studies/runs-2026-07-21-exploratory/opus-5/entrainment-02/theta_lib.py:88: UserWarning: Elements should not be passed as <class 'numpy.ndarray'>. Default time units is seconds when creating the Ts object.
  tsgroup = nap.TsGroup(spikes)


/Users/bdichter/dev/agent-foundational-studies/runs-2026-07-21-exploratory/opus-5/entrainment-02/theta_lib.py:88: UserWarning: Elements should not be passed as <class 'numpy.ndarray'>. Default time units is seconds when creating the Ts object.
  tsgroup = nap.TsGroup(spikes)


alignment: {'lfp_duration': 5471.352, 'last_spike': 5471.30555, 'ratio': 0.9999915103250532}
units: 109 curated | theta reference: electrode row 63 -> LFP column 63
behaviour: 1.3-1754.6 s, median speed 15.3 cm/s
trials: 168, cooling states {np.str_('Cooling off'): np.int64(24), np.str_('Cooling on'): np.int64(29), np.str_('Post-Cooling'): np.int64(76), np.str_('Pre-Cooling'): np.int64(39)}
LFP: 2191640 samples @ 1250.0 Hz, 1.3-1754.6 s, range -6.39 to 6.39 mV
running epochs: 68, total 1607 s (92% of the behavioural period)
wrote fig01_raw_data_validation.png
wrote fig02_power_spectrum.png


![raw data validation](fig01_raw_data_validation.png)

The wideband LFP shows a continuous ~8 Hz oscillation during running, the extracted
phase advances monotonically through each cycle, and the septal temperature trace
shows the cooling epoch (37 °C down to 20 °C and back) in the middle of the session.

![power spectrum](fig02_power_spectrum.png)

The theta peak is present during maze running and absent during post-maze rest, and
theta amplitude grows with running speed. Both are the standard checks that the
5-11 Hz band on this channel is hippocampal theta rather than filter ringing.

## 2. Phase locking, session by session

For each of the five sessions, every curated unit with at least 200 spikes in the
running epochs gets a spike-phase distribution, an MRL, a preferred phase, a Rayleigh
test, and a 200-sample circular-shift null. Units are also split into putative
pyramidal cells and putative interneurons using firing rate and the 3-5 ms mass of
the autocorrelogram, since this dandiset does not distribute spike waveforms.

Cooled and normal running epochs are separated by septal temperature (>34 °C normal,
<30 °C cooled) so that the main measurement is made on undisturbed theta.

In [4]:
run_step("02_run_all_sessions.py")

--- 02_run_all_sessions.py ---


Peter-MS21-180628-155921-concat: loaded from cache (100 units)
Peter-MS21-180629-110332-concat: loaded from cache (139 units)
Peter-MS22-180629-110319-concat: loaded from cache (121 units)
Peter-MS21-180625-153927-concat: loaded from cache (73 units)
Peter-MS13-171128-113924-concat: loaded from cache (50 units)

483 units from 5 sessions
               n       mrl       rate     burst
cell_type                                      
interneuron   39  0.095472  26.445667  0.735141
pyramidal    444  0.069120   1.390881  2.320118
significant (shuffle p<0.05): 310 / 483


In [5]:
df = pd.read_csv("results/unit_stats.csv")
print(f"{len(df)} units from {df.session.nunique()} sessions")
print(df.groupby("cell_type").agg(n=("mrl", "size"), median_rate=("rate", "median"),
                                  median_burst=("burst", "median"),
                                  median_mrl=("mrl", "median"),
                                  median_null_mrl=("mrl_null_mean", "median")))

483 units from 5 sessions
               n  median_rate  median_burst  median_mrl  median_null_mrl
cell_type                                                               
interneuron   39    26.445667      0.735141    0.095472         0.006541
pyramidal    444     1.390881      2.320118    0.069120         0.026564


## 3. Population result

The observed MRLs sit far above the shuffled distribution, and about two thirds of
units are individually significant. Sorting every locked unit by its preferred phase
shows that preferred phases tile the theta cycle rather than piling up at one value,
with a population-level bias.

In [6]:
run_step("03_population.py")

--- 03_population.py ---


483 units, 310 phase-locked (circular-shift shuffle, p<0.05)
wrote fig03_example_units.png
wrote fig04_population_summary.png
wrote fig05_population_heatmap.png
{'n_units': 483, 'n_sessions': 5, 'n_locked': 310, 'frac_locked': 0.6418219461697723, 'median_mrl': 0.0731811978399218, 'median_mrl_null': 0.0252926347011313, 'median_mrl_pyr': 0.0596070706929899, 'median_mrl_int': 0.0953354273882831, 'pref_phase_diff_deg': 62.43531478135811, 'pref_phase_perm_p': 0.0003999200159968006, 'pref_phase_rayleigh_p_pyr': 1.5535013720088186e-26, 'pref_phase_rayleigh_p_int': 1.8909845859533382e-06, 'pref_phase_pyr': 240.50612501179836, 'pref_phase_int': 178.0708102304403, 'mannwhitney_p': 1.858674109382881e-06}


![example units](fig03_example_units.png)

Individual units, from the most strongly modulated to two that fail the test. The
blue curve is the theta cycle for reference and the dashed line marks the preferred
phase.

![population summary](fig04_population_summary.png)

Top left: the observed MRL distribution against the circular-shift null. Top middle:
each unit against its own 95th-percentile null, which is the test that generates the
significance count. Bottom left: preferred phases of locked units, separated by cell
type. Putative interneurons lock more strongly than putative pyramidal cells and
prefer an earlier phase.

![population heatmap](fig05_population_heatmap.png)

Every significantly locked unit, normalised to its own mean rate and sorted by
preferred phase, over two theta cycles.

In [7]:
summary = pd.read_json("results/summary.json", typ="series")
print(summary)

n_units                      4.830000e+02
n_sessions                   5.000000e+00
n_locked                     3.100000e+02
frac_locked                  6.418219e-01
median_mrl                   7.318120e-02
median_mrl_null              2.529263e-02
median_mrl_pyr               5.960707e-02
median_mrl_int               9.533543e-02
pref_phase_diff_deg          6.243531e+01
pref_phase_perm_p            3.999200e-04
pref_phase_rayleigh_p_pyr    1.553501e-26
pref_phase_rayleigh_p_int    1.891000e-06
pref_phase_pyr               2.405061e+02
pref_phase_int               1.780708e+02
mannwhitney_p                1.858700e-06
dtype: float64


## 4. Is the effect an artefact?

Three checks. The spike-triggered average of the raw LFP should oscillate at theta
for locked units and stay inside its own circular-shift band for unlocked ones. The
preferred phase should repeat across interleaved subsets of the running epochs rather
than being a property of one stretch of the recording. And the phase preference
should not be explained by speed or by the unit's own spike history.

In [8]:
run_step("04_examples.py")

--- 04_examples.py ---


/Users/bdichter/dev/agent-foundational-studies/runs-2026-07-21-exploratory/opus-5/entrainment-02/theta_lib.py:88: UserWarning: Elements should not be passed as <class 'numpy.ndarray'>. Default time units is seconds when creating the Ts object.
  tsgroup = nap.TsGroup(spikes)


/Users/bdichter/dev/agent-foundational-studies/runs-2026-07-21-exploratory/opus-5/entrainment-02/theta_lib.py:88: UserWarning: Elements should not be passed as <class 'numpy.ndarray'>. Default time units is seconds when creating the Ts object.
  tsgroup = nap.TsGroup(spikes)


/Users/bdichter/dev/agent-foundational-studies/runs-2026-07-21-exploratory/opus-5/entrainment-02/theta_lib.py:88: UserWarning: Elements should not be passed as <class 'numpy.ndarray'>. Default time units is seconds when creating the Ts object.
  tsgroup = nap.TsGroup(spikes)


Peter-MS21-180628-155921-concat: 109 curated units, run 1607 s (normal 1263 s, cooled 246 s)
wrote fig06_sta_and_stability.png


![STA and stability](fig06_sta_and_stability.png)

Left: spike-triggered LFP averages with the 5-95% range of circular-shift shuffles
shaded. The three locked units produce theta-rhythmic averages many times larger than
chance; the two units with the weakest locking stay near their bands. Middle: the same
unit's phase histogram computed on odd and on even running epochs. Right: preferred
phase on odd versus even epochs for every unit with enough spikes, pooled over
sessions.

### A GLM control for speed and spike history

A concentrated spike-phase distribution can in principle arise without entrainment:
a unit that bursts at roughly theta frequency has autocorrelated spike times, and
running speed modulates both firing rate and theta. The Poisson GLM below always
contains a spline basis over running speed and a raised-cosine basis over the unit's
own spike history; the question is how much held-out log-likelihood is gained by
adding a cyclic spline basis over theta phase.

In [9]:
run_step("06_glm.py")

--- 06_glm.py ---


/Users/bdichter/dev/agent-foundational-studies/runs-2026-07-21-exploratory/opus-5/entrainment-02/theta_lib.py:88: UserWarning: Elements should not be passed as <class 'numpy.ndarray'>. Default time units is seconds when creating the Ts object.
  tsgroup = nap.TsGroup(spikes)


Peter-MS21-180628-155921-concat: 109 curated units, run 1607 s (normal 1263 s, cooled 246 s)


GLM per unit:   0%|          | 0/40 [00:00<?, ?it/s]

/Users/bdichter/miniconda3/lib/python3.12/site-packages/nemos/convolve.py:671: UserWarning: One or more trials are shorter than the convolution window size (25 samples). These trials will produce NaN values in the output.
  validation._check_trials_longer_than_time_window(


...

GLM per unit:  92%|█████████▎| 37/40 [04:54<00:22,  7.34s/it]/Users/bdichter/miniconda3/lib/python3.12/site-packages/nemos/convolve.py:671: UserWarning: One or more trials are shorter than the convolution window size (25 samples). These trials will produce NaN values in the output.
  validation._check_trials_longer_than_time_window(
/Users/bdichter/miniconda3/lib/python3.12/site-packages/pynapple/core/utils.py:198: UserWarning: Converting 'd' to numpy.array. The provided array was of type 'ArrayImpl'.
  warnings.warn(


GLM per unit:  95%|█████████▌| 38/40 [05:02<00:14,  7.38s/it]/Users/bdichter/miniconda3/lib/python3.12/site-packages/nemos/convolve.py:671: UserWarning: One or more trials are shorter than the convolution window size (25 samples). These trials will produce NaN values in the output.
  validation._check_trials_longer_than_time_window(
/Users/bdichter/miniconda3/lib/python3.12/site-packages/pynapple/core/utils.py:198: UserWarning: Converting 'd' to numpy.array. The provided array was of type 'ArrayImpl'.
  warnings.warn(


GLM per unit:  98%|█████████▊| 39/40 [05:09<00:07,  7.25s/it]/Users/bdichter/miniconda3/lib/python3.12/site-packages/nemos/convolve.py:671: UserWarning: One or more trials are shorter than the convolution window size (25 samples). These trials will produce NaN values in the output.
  validation._check_trials_longer_than_time_window(
/Users/bdichter/miniconda3/lib/python3.12/site-packages/pynapple/core/utils.py:198: UserWarning: Converting 'd' to numpy.array. The provided array was of type 'ArrayImpl'.
  warnings.warn(


GLM per unit: 100%|██████████| 40/40 [05:17<00:00,  7.93s/it]


              unit  d_ll_per_spike  ...  shuffle_p      n_spikes
count    40.000000       40.000000  ...  40.000000     40.000000
mean   1765.775000        0.004187  ...   0.032711  13681.725000
std     435.703292        0.006965  ...   0.064129  15575.222649
min      36.000000       -0.001325  ...   0.004975   3498.000000
25%    1686.500000        0.000201  ...   0.004975   4692.250000
50%    1960.500000        0.002155  ...   0.004975   6426.500000
75%    2001.750000        0.005119  ...   0.031095  14048.250000
max    2089.000000        0.037977  ...   0.318408  66342.000000

[8 rows x 8 columns]
wrote fig08_glm.png


![GLM](fig08_glm.png)

In [10]:
glm = pd.read_csv("results/glm_results.csv")
print(f"{(glm.d_ll_per_spike > 0).sum()}/{len(glm)} units improve with the phase term; "
      f"median Δ log-likelihood per spike = {glm.d_ll_per_spike.median():.4f}")

31/40 units improve with the phase term; median Δ log-likelihood per spike = 0.0022


## 5. A causal manipulation: cooling the medial septum

Cooling the septum slows the theta rhythm. If the spike-phase distributions measured
above reflect entrainment to that rhythm, they should track it when its frequency
changes rather than disappearing.

The comparison has to be matched. Cooled running time is shorter than normal running
time, and the MRL is biased upward both by small spike counts and by short observation
windows. Each unit's normal-condition MRL is therefore computed inside contiguous
blocks of normal running epochs whose total duration equals the cooled duration, and
then subsampled to the same spike count.

In [11]:
run_step("05_cooling.py")

--- 05_cooling.py ---


/Users/bdichter/dev/agent-foundational-studies/runs-2026-07-21-exploratory/opus-5/entrainment-02/theta_lib.py:88: UserWarning: Elements should not be passed as <class 'numpy.ndarray'>. Default time units is seconds when creating the Ts object.
  tsgroup = nap.TsGroup(spikes)


Peter-MS21-180628-155921-concat: 109 curated units, run 1607 s (normal 1263 s, cooled 246 s)


Peter-MS21-180628-155921-concat cooling:  28%|██▊       | 31/109 [00:00<00:00, 250.32it/s]

Peter-MS21-180628-155921-concat cooling:  70%|██████▉   | 76/109 [00:00<00:00, 176.58it/s]

...

Peter-MS21-180625-153927-concat cooling: 100%|██████████| 83/83 [00:00<00:00, 258.92it/s]


Peter-MS13-171128-113924-concat: 73 curated units, run 2240 s (normal 1354 s, cooled 665 s)


Peter-MS13-171128-113924-concat cooling:  33%|███▎      | 24/73 [00:00<00:00, 193.61it/s]

Peter-MS13-171128-113924-concat cooling: 100%|██████████| 73/73 [00:00<00:00, 237.79it/s]


380 units with enough spikes in both conditions
wrote fig07_cooling.png
{
 "n_units":380.0,
 "freq_normal":8.1168831169,
 "freq_cooled":7.4404761905,
 "freq_delta":-0.6550653146,
 "env_ratio":0.8170703149,
 "mrl_normal_matched":0.0819550124,
 "mrl_cooled_matched":0.117402256,
 "frac_above_unity":0.7236842105,
 "wilcoxon_p":8.643126892e-24,
 "median_abs_phase_shift_deg":21.9357258546,
 "rate_ratio":0.9045538247
}


![cooling](fig07_cooling.png)

In [12]:
cool = pd.read_json("results/cooling_summary.json", typ="series")
print(cool)

n_units                       3.800000e+02
freq_normal                   8.116883e+00
freq_cooled                   7.440476e+00
freq_delta                   -6.550653e-01
env_ratio                     8.170703e-01
mrl_normal_matched            8.195501e-02
mrl_cooled_matched            1.174023e-01
frac_above_unity              7.236842e-01
wilcoxon_p                    8.643127e-24
median_abs_phase_shift_deg    2.193573e+01
rate_ratio                    9.045538e-01
dtype: float64


## 6. What the data show

Hippocampal spiking during locomotion is locked to LFP theta phase. Across five
sessions, about two thirds of curated units have spike-phase distributions that are
individually significant against a circular-shift null that preserves each unit's
spike count and burst structure. The effect is present in the spike-triggered LFP
average, repeats across interleaved subsets of each session, and survives a GLM that
already contains running speed and spike history.

Putative interneurons are more strongly entrained than putative pyramidal cells and
fire at an earlier phase. Cooling the medial septum slows theta by roughly 0.7 Hz and
reduces its amplitude, and phase locking not only survives but strengthens, with
preferred phases largely preserved. The measurement therefore tracks the oscillation
itself rather than any fixed timing in the recording.